# SiT failure debug: inside every Transformer block

For rank 0 and rank 1 independently:

1. Load the saved pre-failure model and failing SiT input.
2. Run one forward pass.
3. Trace every Transformer block and its important internal layers.
4. Backpropagate once and trace the corresponding activation gradients.
5. Print forward/backward RMS, absmax, and gain.

For each block we inspect:

`Block input -> adaLN -> norm1 -> qkv -> attention -> attention proj -> norm2 -> MLP fc1 -> MLP fc2 -> Block output`

We also print the six adaLN components:

`shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp`

and the actual forward residual-branch magnitudes:

`gate_msa * attention_output` and `gate_mlp * mlp_output`.

In [ ]:
import os
import sys
from pathlib import Path
import torch

REPO_ROOT = "/vol/biomedic3/tx1215/REPA-E"
FAILURE_DIR = "/vol/biomedic3/tx1215/REPA-E/checkpoints/embed_latent_dit_sits2_256x192_debug/failures/step_0037386"
DEVICE = torch.device("cuda:0")

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from models.sit import SiT_models

def load_pt(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

ckpt = load_pt(Path(FAILURE_DIR) / "model_before_failure.pt")
rank0 = load_pt(Path(FAILURE_DIR) / "rank_0.pt")
rank1 = load_pt(Path(FAILURE_DIR) / "rank_1.pt")
args = ckpt["args"]

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("failing step:", ckpt.get("failing_step"))
print("rank0 bad gradients:", rank0.get("bad_grad_stats"))
print("rank1 bad gradients:", rank1.get("bad_grad_stats"))

In [ ]:
def getarg(name, default=None):
    return getattr(args, name, default)

def move_to(x):
    if torch.is_tensor(x):
        return x.to(DEVICE)
    if isinstance(x, dict):
        return {k: move_to(v) for k, v in x.items()}
    if isinstance(x, list):
        return [move_to(v) for v in x]
    if isinstance(x, tuple):
        return tuple(move_to(v) for v in x)
    return x

def first_tensor(x):
    if torch.is_tensor(x):
        return x
    if isinstance(x, (list, tuple)):
        for v in x:
            t = first_tensor(v)
            if t is not None:
                return t
    if isinstance(x, dict):
        for v in x.values():
            t = first_tensor(v)
            if t is not None:
                return t
    return None

def stats(x):
    x = x.detach()
    finite = torch.isfinite(x)
    xf = x[finite].float()
    return {
        "shape": tuple(x.shape),
        "rms": torch.sqrt((xf ** 2).mean()).item() if xf.numel() else float("nan"),
        "std": xf.std(unbiased=False).item() if xf.numel() else float("nan"),
        "absmax": xf.abs().max().item() if xf.numel() else float("nan"),
        "nan": torch.isnan(x).sum().item(),
        "inf": torch.isinf(x).sum().item(),
    }

def restore_rng(snapshot):
    state = snapshot.get("forward_rng_state")
    if state is None:
        return
    if state.get("cpu") is not None:
        torch.set_rng_state(state["cpu"])
    if state.get("cuda") is not None:
        torch.cuda.set_rng_state(state["cuda"], device=DEVICE)

def precision_context(snapshot):
    precision = snapshot.get("mixed_precision", getarg("mixed_precision", "bf16"))
    if precision == "bf16":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    if precision == "fp16":
        return torch.autocast("cuda", dtype=torch.float16)
    return torch.autocast("cuda", enabled=False)

def infer_z_dims(state):
    z_dims = []
    i = 0
    while f"projectors.{i}.4.weight" in state:
        z_dims.append(state[f"projectors.{i}.4.weight"].shape[0])
        i += 1
    return z_dims

In [ ]:
def build_model(snapshot):
    z = snapshot["z"]
    latent_size = tuple(z.shape[-2:])
    in_channels = z.shape[1]
    state = ckpt["model"]

    model = SiT_models[getarg("model")](
        input_size=latent_size,
        in_channels=in_channels,
        num_classes=getarg("num_classes", 1000),
        class_dropout_prob=getarg("cfg_prob", 0.1),
        z_dims=infer_z_dims(state),
        encoder_depth=getarg("encoder_depth", 8),
        bn_momentum=getarg("bn_momentum", 0.1),
        fused_attn=snapshot.get("fused_attn", getarg("fused_attn", True)),
        qk_norm=snapshot.get("qk_norm", getarg("qk_norm", False)),
    )

    model.load_state_dict(state)
    model = model.to(DEVICE)
    model.train()
    model.bn.eval()
    return model

In [ ]:
def register_hooks(model):
    trace = {}
    handles = []
    adaln_full = {}

    def hook_layer(name, module):
        def hook(module, inputs, output):
            x = first_tensor(inputs)
            y = first_tensor(output)
            entry = trace.setdefault(name, {})
            if x is not None:
                entry["forward_in"] = stats(x)
                if x.requires_grad:
                    x.register_hook(lambda g, n=name: trace[n].update({"grad_in": stats(g)}))
            if y is not None:
                entry["forward_out"] = stats(y)
                if y.requires_grad:
                    y.register_hook(lambda g, n=name: trace[n].update({"grad_out": stats(g)}))
        handles.append(module.register_forward_hook(hook))

    def hook_adaln(block_idx, module):
        name = f"Block {block_idx}.adaLN"
        def hook(module, inputs, output):
            x = first_tensor(inputs)
            entry = trace.setdefault(name, {})
            entry["forward_in"] = stats(x)
            entry["forward_out"] = stats(output)
            chunks = output.chunk(6, dim=-1)
            names = ["shift_msa", "scale_msa", "gate_msa", "shift_mlp", "scale_mlp", "gate_mlp"]
            entry["components"] = {k: stats(v) for k, v in zip(names, chunks)}
            adaln_full[block_idx] = {"gate_msa": chunks[2].detach(), "gate_mlp": chunks[5].detach()}
            if x.requires_grad:
                x.register_hook(lambda g, n=name: trace[n].update({"grad_in": stats(g)}))
            if output.requires_grad:
                def grad_hook(g):
                    trace[name]["grad_out"] = stats(g)
                    gchunks = g.chunk(6, dim=-1)
                    trace[name]["grad_components"] = {k: stats(v) for k, v in zip(names, gchunks)}
                output.register_hook(grad_hook)
        handles.append(module.register_forward_hook(hook))

    def hook_attn(block_idx, module):
        name = f"Block {block_idx}.attn"
        def hook(module, inputs, output):
            x = first_tensor(inputs)
            y = first_tensor(output)
            entry = trace.setdefault(name, {})
            entry["forward_in"] = stats(x)
            entry["forward_out"] = stats(y)
            gate = adaln_full[block_idx]["gate_msa"]
            entry["residual_branch"] = stats(gate.unsqueeze(1) * y.detach())
            if x.requires_grad:
                x.register_hook(lambda g, n=name: trace[n].update({"grad_in": stats(g)}))
            if y.requires_grad:
                y.register_hook(lambda g, n=name: trace[n].update({"grad_out": stats(g)}))
        handles.append(module.register_forward_hook(hook))

    def hook_mlp(block_idx, module):
        name = f"Block {block_idx}.mlp"
        def hook(module, inputs, output):
            x = first_tensor(inputs)
            y = first_tensor(output)
            entry = trace.setdefault(name, {})
            entry["forward_in"] = stats(x)
            entry["forward_out"] = stats(y)
            gate = adaln_full[block_idx]["gate_mlp"]
            entry["residual_branch"] = stats(gate.unsqueeze(1) * y.detach())
            if x.requires_grad:
                x.register_hook(lambda g, n=name: trace[n].update({"grad_in": stats(g)}))
            if y.requires_grad:
                y.register_hook(lambda g, n=name: trace[n].update({"grad_out": stats(g)}))
        handles.append(module.register_forward_hook(hook))

    hook_layer("PatchEmbed", model.x_embedder)
    for i, block in enumerate(model.blocks):
        hook_layer(f"Block {i}", block)
        hook_adaln(i, block.adaLN_modulation)
        hook_layer(f"Block {i}.norm1", block.norm1)
        hook_layer(f"Block {i}.qkv", block.attn.qkv)
        hook_attn(i, block.attn)
        hook_layer(f"Block {i}.attn_proj", block.attn.proj)
        hook_layer(f"Block {i}.norm2", block.norm2)
        hook_layer(f"Block {i}.fc1", block.mlp.fc1)
        hook_layer(f"Block {i}.fc2", block.mlp.fc2)
        hook_mlp(i, block.mlp)
    hook_layer("FinalLayer", model.final_layer)
    return trace, handles

In [ ]:
def run_rank(snapshot, rank_name):
    model = build_model(snapshot)
    trace, handles = register_hooks(model)

    z = move_to(snapshot["z"])
    parents = move_to(snapshot["parents"])
    zs = move_to(snapshot.get("zs"))
    time_input = move_to(snapshot.get("time_input")) if snapshot.get("time_input") is not None else None
    noises = move_to(snapshot.get("noises")) if snapshot.get("noises") is not None else None

    if time_input is None or noises is None:
        restore_rng(snapshot)

    model.zero_grad(set_to_none=True)

    with precision_context(snapshot):
        out = model(
            x=z,
            parents=parents,
            zs=zs,
            loss_kwargs={
                "weighting": getarg("weighting", "uniform"),
                "path_type": getarg("path_type", "linear"),
                "prediction": getarg("prediction", "v"),
                "align_only": False,
            },
            time_input=time_input,
            noises=noises,
        )
        denoise = out["denoising_loss"].mean()
        proj = out["proj_loss"].mean()
        loss = denoise + getarg("proj_coeff", 0.0) * proj

    loss.backward()
    for h in handles:
        h.remove()

    print(f"{rank_name} loss: {loss.item():.8g}")
    print("saved loss:", snapshot.get("losses", {}).get("sit_loss"))
    return model, trace

In [ ]:
def ratio(a, b):
    return a / b if b is not None and b > 0 else float("nan")

def fmt(x):
    return "N/A" if x is None or x != x else f"{x:.4g}"

def print_layer(name, t, indent=""):
    fi = t.get("forward_in")
    fo = t.get("forward_out")
    gi = t.get("grad_in")
    go = t.get("grad_out")
    fwd_gain = ratio(fo["rms"], fi["rms"]) if fi is not None and fo is not None else float("nan")
    bwd_gain = ratio(gi["rms"], go["rms"]) if gi is not None and go is not None else float("nan")
    bad = 0
    for s in [fi, fo, gi, go]:
        if s is not None:
            bad += s["nan"] + s["inf"]
    print(
        f"{indent}{name:18s} "
        f"{fmt(fi['rms'] if fi else None):>10s} "
        f"{fmt(fo['rms'] if fo else None):>10s} "
        f"{fmt(fwd_gain):>10s} "
        f"{fmt(go['rms'] if go else None):>10s} "
        f"{fmt(gi['rms'] if gi else None):>10s} "
        f"{fmt(bwd_gain):>10s} "
        f"{fmt(fo['absmax'] if fo else None):>10s} "
        f"{bad:6d}"
    )

def print_deep_trace(trace, title):
    print(f"\n===== {title} =====")
    print(
        f"{'Layer':20s} "
        f"{'In RMS':>10s} {'Out RMS':>10s} {'Fwd gain':>10s} "
        f"{'Grad out':>10s} {'Grad in':>10s} {'Bwd gain':>10s} "
        f"{'Out max':>10s} {'bad':>6s}"
    )
    print_layer("PatchEmbed", trace["PatchEmbed"])
    block_ids = sorted(int(k.split()[1]) for k in trace if k.startswith("Block ") and "." not in k)
    for i in block_ids:
        print(f"\n--- Block {i} ---")
        print_layer("BLOCK", trace[f"Block {i}"], indent="  ")
        print_layer("adaLN", trace[f"Block {i}.adaLN"], indent="  ")
        print_layer("norm1", trace[f"Block {i}.norm1"], indent="  ")
        print_layer("qkv", trace[f"Block {i}.qkv"], indent="  ")
        print_layer("attention", trace[f"Block {i}.attn"], indent="  ")
        print_layer("attn_proj", trace[f"Block {i}.attn_proj"], indent="  ")
        print_layer("norm2", trace[f"Block {i}.norm2"], indent="  ")
        print_layer("fc1", trace[f"Block {i}.fc1"], indent="  ")
        print_layer("fc2", trace[f"Block {i}.fc2"], indent="  ")
        print_layer("mlp", trace[f"Block {i}.mlp"], indent="  ")

        a = trace[f"Block {i}.adaLN"]["components"]
        print("  adaLN components:")
        for name in ["shift_msa", "scale_msa", "gate_msa", "shift_mlp", "scale_mlp", "gate_mlp"]:
            s = a[name]
            print(f"    {name:10s} RMS={s['rms']:10.4g} absmax={s['absmax']:10.4g} bad={s['nan'] + s['inf']}")

        attn_branch = trace[f"Block {i}.attn"]["residual_branch"]
        mlp_branch = trace[f"Block {i}.mlp"]["residual_branch"]
        print(
            f"  residual branches: attn RMS={attn_branch['rms']:.4g}, absmax={attn_branch['absmax']:.4g} | "
            f"mlp RMS={mlp_branch['rms']:.4g}, absmax={mlp_branch['absmax']:.4g}"
        )

    print("\n--- Final ---")
    print_layer("FinalLayer", trace["FinalLayer"], indent="  ")

In [ ]:
model0, trace0 = run_rank(rank0, "Rank 0")
print_deep_trace(trace0, "RANK 0")

In [ ]:
del model0
torch.cuda.empty_cache()

model1, trace1 = run_rank(rank1, "Rank 1")
print_deep_trace(trace1, "RANK 1")

## Conditioning / adaLN diagnosis

The previous trace shows that LayerNorm itself stays well behaved (`RMS ≈ 1`), while the adaLN-conditioned branches become extremely large.

Now check three things directly:

1. `t_embed`
2. `parent_embed`
3. `c = t_embed + parent_embed`

Then inspect the learned `adaLN_modulation` linear weights and biases in every block.

Interpretation:

- large `t_embed` → timestep-conditioning pathway has grown
- large `parent_embed` → parent-conditioning pathway has grown
- moderate `c` but huge adaLN outputs → adaLN linear weights/biases have grown
- both large → both pathways are contributing

In [ ]:
def module_tensor_stats(x):
    s = stats(x)
    return f"RMS={s['rms']:.6g}, std={s['std']:.6g}, absmax={s['absmax']:.6g}, nan={s['nan']}, inf={s['inf']}"

def register_condition_hooks(model):
    captured = {}
    handles = []

    def save_output(name):
        def hook(module, inputs, output):
            captured[name] = output.detach().clone()
        return hook

    def save_input(name):
        def hook(module, inputs, output):
            captured[name] = inputs[0].detach().clone()
        return hook

    handles.append(model.t_embedder.register_forward_hook(save_output("t_embed")))
    handles.append(model.parent_embedder.register_forward_hook(save_output("parent_embed")))

    # Every block receives the same conditioning vector c.
    # The input to the first adaLN_modulation is exactly c.
    handles.append(model.blocks[0].adaLN_modulation.register_forward_hook(save_input("c")))

    return captured, handles

In [ ]:
def inspect_adaln_parameters(model):
    rows = []

    for i, block in enumerate(model.blocks):
        linear = None

        # adaLN_modulation is normally Sequential(SiLU, Linear)
        for module in reversed(list(block.adaLN_modulation.modules())):
            if isinstance(module, torch.nn.Linear):
                linear = module
                break

        if linear is None:
            raise RuntimeError(f"No Linear layer found in Block {i} adaLN_modulation")

        w = stats(linear.weight)
        b = stats(linear.bias) if linear.bias is not None else None

        rows.append({
            "block": i,
            "weight_rms": w["rms"],
            "weight_absmax": w["absmax"],
            "bias_rms": b["rms"] if b else float("nan"),
            "bias_absmax": b["absmax"] if b else float("nan"),
        })

    print(
        f"{'Block':>5s} "
        f"{'W RMS':>12s} {'W absmax':>12s} "
        f"{'b RMS':>12s} {'b absmax':>12s}"
    )

    for r in rows:
        print(
            f"{r['block']:5d} "
            f"{r['weight_rms']:12.6g} {r['weight_absmax']:12.6g} "
            f"{r['bias_rms']:12.6g} {r['bias_absmax']:12.6g}"
        )

    return rows

In [ ]:
def run_condition_diagnosis(snapshot, rank_name):
    model = build_model(snapshot)
    captured, handles = register_condition_hooks(model)

    z = move_to(snapshot["z"])
    parents = move_to(snapshot["parents"])
    zs = move_to(snapshot.get("zs"))
    time_input = move_to(snapshot.get("time_input")) if snapshot.get("time_input") is not None else None
    noises = move_to(snapshot.get("noises")) if snapshot.get("noises") is not None else None

    if time_input is None or noises is None:
        restore_rng(snapshot)

    model.zero_grad(set_to_none=True)

    with precision_context(snapshot):
        out = model(
            x=z,
            parents=parents,
            zs=zs,
            loss_kwargs={
                "weighting": getarg("weighting", "uniform"),
                "path_type": getarg("path_type", "linear"),
                "prediction": getarg("prediction", "v"),
                "align_only": False,
            },
            time_input=time_input,
            noises=noises,
        )

    for h in handles:
        h.remove()

    print(f"\n===== {rank_name}: CONDITIONING =====")
    print("t_embed     :", module_tensor_stats(captured["t_embed"]))
    print("parent_embed:", module_tensor_stats(captured["parent_embed"]))
    print("c           :", module_tensor_stats(captured["c"]))

    # Verify that the captured c is actually t_embed + parent_embed.
    diff = (captured["c"].float() - (captured["t_embed"].float() + captured["parent_embed"].float())).abs()
    print("max |c - (t_embed + parent_embed)| =", diff.max().item())

    print("\n===== adaLN LINEAR PARAMETERS =====")
    adaln_rows = inspect_adaln_parameters(model)

    print("\n===== adaLN OUTPUT AMPLIFICATION =====")
    c_rms = stats(captured["c"])["rms"]
    print(f"{'Block':>5s} {'adaLN out RMS':>15s} {'out/c gain':>12s} {'out absmax':>12s}")

    with torch.no_grad():
        for i, block in enumerate(model.blocks):
            with precision_context(snapshot):
                y = block.adaLN_modulation(captured["c"])
            s = stats(y)
            gain = s["rms"] / c_rms if c_rms > 0 else float("nan")
            print(f"{i:5d} {s['rms']:15.6g} {gain:12.6g} {s['absmax']:12.6g}")

    return model, captured, adaln_rows

In [ ]:
cond_model0, cond0, adaln0 = run_condition_diagnosis(rank0, "RANK 0")

In [ ]:
del cond_model0
torch.cuda.empty_cache()

cond_model1, cond1, adaln1 = run_condition_diagnosis(rank1, "RANK 1")

## How to read this

The key comparison is:

`parent_embed RMS` vs `t_embed RMS` vs `c RMS`

and then:

`c RMS → adaLN output RMS`

If `parent_embed` is near unit scale but `t_embed` is around 15, the timestep pathway dominates `c`.

If `c` is around 15 but some adaLN outputs are tens or hundreds, inspect the corresponding adaLN linear weights/biases.

This section deliberately does not change precision, attention implementation, or model behavior. It only measures the saved pre-failure state.

## Timestep embedding diagnosis

Trace the timestep-conditioning pathway:

`t → fixed sinusoidal embedding → Linear 1 → SiLU → Linear 2 → t_embed`

The input to the first Linear layer is the fixed sinusoidal timestep representation, so this shows exactly where the magnitude first becomes large. We also inspect the two timestep-MLP Linear weights and biases.

In [ ]:
def register_tembed_hooks(model):
    captured = {}
    handles = []

    def save_io(name):
        def hook(module, inputs, output):
            x = first_tensor(inputs)
            y = first_tensor(output)
            captured[name] = {
                "input": x.detach().clone() if x is not None else None,
                "output": y.detach().clone() if y is not None else None,
            }
        return hook

    handles.append(model.t_embedder.register_forward_hook(save_io("t_embedder")))
    for i, layer in enumerate(model.t_embedder.mlp):
        handles.append(layer.register_forward_hook(save_io(f"mlp.{i}")))

    return captured, handles

def inspect_tembed_parameters(model):
    print("\n===== TIMESTEP MLP PARAMETERS =====")
    print(f"{'Layer':12s} {'W RMS':>12s} {'W absmax':>12s} {'b RMS':>12s} {'b absmax':>12s}")
    rows = []
    for i, layer in enumerate(model.t_embedder.mlp):
        if not isinstance(layer, torch.nn.Linear):
            continue
        w = stats(layer.weight)
        b = stats(layer.bias) if layer.bias is not None else None
        row = {
            "name": f"mlp.{i}",
            "weight_rms": w["rms"],
            "weight_absmax": w["absmax"],
            "bias_rms": b["rms"] if b else float("nan"),
            "bias_absmax": b["absmax"] if b else float("nan"),
        }
        rows.append(row)
        print(
            f"{row['name']:12s} "
            f"{row['weight_rms']:12.6g} {row['weight_absmax']:12.6g} "
            f"{row['bias_rms']:12.6g} {row['bias_absmax']:12.6g}"
        )
    return rows

In [ ]:
def run_tembed_diagnosis(snapshot, rank_name):
    model = build_model(snapshot)
    captured, handles = register_tembed_hooks(model)

    z = move_to(snapshot["z"])
    parents = move_to(snapshot["parents"])
    zs = move_to(snapshot.get("zs"))
    time_input = move_to(snapshot.get("time_input")) if snapshot.get("time_input") is not None else None
    noises = move_to(snapshot.get("noises")) if snapshot.get("noises") is not None else None

    if time_input is None or noises is None:
        restore_rng(snapshot)

    model.zero_grad(set_to_none=True)

    with precision_context(snapshot):
        _ = model(
            x=z,
            parents=parents,
            zs=zs,
            loss_kwargs={
                "weighting": getarg("weighting", "uniform"),
                "path_type": getarg("path_type", "linear"),
                "prediction": getarg("prediction", "v"),
                "align_only": False,
            },
            time_input=time_input,
            noises=noises,
        )

    for h in handles:
        h.remove()

    print(f"\n===== {rank_name}: TIMESTEP PATH =====")
    raw_t = captured["t_embedder"]["input"]
    t_freq = captured["mlp.0"]["input"]
    t_embed = captured["t_embedder"]["output"]

    print("raw t               :", module_tensor_stats(raw_t))
    print("sin/cos embedding   :", module_tensor_stats(t_freq))
    for i in range(len(model.t_embedder.mlp)):
        key = f"mlp.{i}"
        if key in captured:
            print(f"{key:20s}: {module_tensor_stats(captured[key]['output'])}")
    print("final t_embed       :", module_tensor_stats(t_embed))

    freq_rms = stats(t_freq)["rms"]
    temb_rms = stats(t_embed)["rms"]
    print("t_embed / sincos RMS gain =", temb_rms / freq_rms if freq_rms > 0 else float("nan"))

    rows = inspect_tembed_parameters(model)
    return model, captured, rows

In [ ]:
temb_model0, temb0, temb_params0 = run_tembed_diagnosis(rank0, "RANK 0")

In [ ]:
del temb_model0
torch.cuda.empty_cache()
temb_model1, temb1, temb_params1 = run_tembed_diagnosis(rank1, "RANK 1")

## Interpretation

Focus on:

`sin/cos embedding RMS → Linear1 → SiLU → Linear2 → t_embed RMS`

If the sin/cos input is normal but one Linear layer causes a large jump, the learned timestep MLP is the source. If `t_embed` stays moderate, then the main amplification is downstream in the block-specific adaLN layers.

## What to inspect first

Start with **Block 0**, because the previous trace showed roughly `0.75 -> 300` RMS there.

Look for the first large jump among:

- adaLN component magnitudes
- `norm1 -> qkv`
- attention output
- `gate_msa * attention_output`
- `norm2 -> fc1 -> fc2`
- `gate_mlp * mlp_output`

Then compare the same internal pattern across Blocks 1-11.